# Notebook 3 — Service Variability: M/D/1 vs M/M/1

**Series:** DES Playground · Discrete Event Simulation in SimPy  
**Prerequisites:** Notebooks 1–2

**What you'll learn:**
- Why variability *directly* worsens queue performance, independent of load
- The **Pollaczek–Khinchine (P-K) formula** linking CV² to queue length
- Compare Constant / Uniform / Exponential / Triangular service distributions
- Measure the **coefficient of variation (CV)** and confirm P-K in simulation

---

## 1 · Background — Why Variability Hurts

Consider two systems at identical load ρ = 0.8:
- **M/D/1**: exponential arrivals, *constant* (deterministic) service → Lq = 1.6
- **M/M/1**: exponential arrivals, exponential service → Lq = 3.2

Same load, **double the queue** — purely because of service-time randomness.

The **P-K formula** (for M/G/1, any service distribution G) is:
```
Lq = ρ² / (1 − ρ)  ·  (1 + CV²) / 2

where  CV = σ_s / E[s]  (coefficient of variation of service time)
```

| Distribution | CV  | Relative Lq vs M/D/1 |
|-------------|-----|---------------------|
| Constant    | 0   | 1× (minimum) |
| Uniform     | 1/√3 ≈ 0.58 | ~1.17× |
| Exponential | 1   | 2× |
| Hyperexponential | >1 | >2× |

In [ ]:
!pip install simpy --quiet

In [ ]:
import simpy
import random
import math
import statistics
from typing import List, Dict, Callable
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

T_TABLE = {1:12.706,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,
           8:2.306,9:2.262,10:2.228,15:2.131,20:2.086,25:2.060,30:2.042}

def t_crit(df):
    if df in T_TABLE: return T_TABLE[df]
    if df > 30: return 1.960
    keys = sorted(T_TABLE)
    for i in range(len(keys)-1):
        lo, hi = keys[i], keys[i+1]
        if lo < df < hi:
            return T_TABLE[lo] + (T_TABLE[hi]-T_TABLE[lo])*(df-lo)/(hi-lo)
    return 2.0

def ci(values):
    n = len(values)
    if n < 2: return dict(mean=values[0] if values else 0, half=0, lo=0, hi=0)
    mu = statistics.mean(values)
    s  = statistics.stdev(values)
    h  = t_crit(n-1) * s / math.sqrt(n)
    return dict(mean=mu, half=h, lo=mu-h, hi=mu+h)

print('Setup complete.')

## 2 · Service Time Distributions

We define four samplers with the **same mean** (1/μ = 5 time units) but
different shapes and therefore different CV values.

In [ ]:
MEAN_SVC = 5.0  # all distributions share this mean
MU       = 1.0 / MEAN_SVC

def sample_constant(rng, mean=MEAN_SVC):
    return mean  # CV = 0

def sample_uniform(rng, mean=MEAN_SVC):
    # Uniform[0, 2*mean] → CV = 1/sqrt(3) ≈ 0.577
    return rng.uniform(0, 2 * mean)

def sample_exponential(rng, mean=MEAN_SVC):
    return rng.expovariate(1.0 / mean)  # CV = 1

def sample_triangular(rng, mean=MEAN_SVC):
    # Triangular(low=1, mode=mean, high=2*mean-1) → roughly CV ≈ 0.37
    low, high, mode = 0.5, 2*mean - 0.5, mean
    return rng.triangular(low, high, mode)

# Measure CV empirically
rng_test = random.Random(0)
for name, fn in [('Constant', sample_constant), ('Triangular', sample_triangular),
                 ('Uniform', sample_uniform), ('Exponential', sample_exponential)]:
    samples = [fn(rng_test) for _ in range(100_000)]
    mu  = statistics.mean(samples)
    std = statistics.stdev(samples)
    cv  = std / mu
    print(f'{name:<14}  mean={mu:.3f}  std={std:.3f}  CV={cv:.3f}')

### Distribution Shape Comparison

In [ ]:
rng_plot = random.Random(1)
dists = [
    ('Constant (CV=0)',        [sample_constant(rng_plot)   for _ in range(5_000)], 'steelblue'),
    ('Triangular (CV≈0.37)',   [sample_triangular(rng_plot) for _ in range(5_000)], 'green'),
    ('Uniform (CV≈0.58)',      [sample_uniform(rng_plot)    for _ in range(5_000)], 'orange'),
    ('Exponential (CV=1)',     [sample_exponential(rng_plot)for _ in range(5_000)], 'red'),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
for ax, (name, samples, color) in zip(axes, dists):
    ax.hist(samples, bins=40, color=color, alpha=0.75, density=True)
    ax.axvline(MEAN_SVC, color='black', ls='--', lw=1.5, label=f'mean={MEAN_SVC}')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Service time')
    if ax == axes[0]: ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Service time distributions — same mean, different variability', fontsize=12)
plt.tight_layout()
plt.show()

## 3 · Run Simulations for Each Distribution

In [ ]:
def run_mg1(
    arrival_rate:  float,
    svc_sampler:   Callable,
    horizon:       float = 30_000,
    warmup:        float = 1_000,
    seed:          int   = 42,
) -> Dict:
    rng = random.Random(seed)
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=1)

    last_t = [0.0]; q_area = [0.0]; b_area = [0.0]
    wait_t: List[float] = []; soj_t: List[float] = []
    deps = [0]

    def integrate(t):
        dt = t - last_t[0]
        q_area[0] += len(srv.queue) * dt
        b_area[0] += srv.count * dt
        last_t[0] = t

    def entity(arr_t):
        track = arr_t >= warmup
        integrate(env.now)
        q_enter = env.now
        with srv.request() as req:
            yield req
            integrate(env.now)
            wait = env.now - q_enter
            svc  = svc_sampler(rng)
            yield env.timeout(svc)
            integrate(env.now)
        deps[0] += 1
        if track:
            wait_t.append(wait)
            soj_t.append(wait + svc)

    def source():
        while True:
            yield env.timeout(rng.expovariate(arrival_rate))
            env.process(entity(env.now))

    env.process(source())
    env.run(until=horizon)
    integrate(horizon)

    obs = max(horizon - warmup, 1.0)
    return {
        'lq':  q_area[0] / obs,
        'rho': b_area[0] / obs,
        'wq':  statistics.mean(wait_t) if wait_t else 0,
        'w':   statistics.mean(soj_t)  if soj_t  else 0,
    }

print('run_mg1() defined.')

In [ ]:
LAMBDA = 0.8 / MEAN_SVC  # ρ ≈ 0.8
RHO    = LAMBDA * MEAN_SVC
N_REPS = 8

print(f'λ={LAMBDA:.4f}, μ={MU:.4f}, ρ={RHO:.2f}')
print()

distributions = [
    ('Constant (M/D/1)',     sample_constant,    0.0),
    ('Triangular (M/G/1)',   sample_triangular,  None),
    ('Uniform (M/G/1)',      sample_uniform,     1/3),
    ('Exponential (M/M/1)', sample_exponential, 1.0),
]

sim_results = {}

print(f'{"Distribution":<24} {"Lq sim":>8} {"±CI":>6} {"Lq P-K":>8} {"Wq sim":>8} {"Wq P-K":>8}')
print('-' * 68)

for name, sampler, cv2 in distributions:
    runs = [run_mg1(LAMBDA, sampler, horizon=40_000, warmup=2_000, seed=42+k*1000)
            for k in range(N_REPS)]
    lq_ci = ci([r['lq'] for r in runs])
    wq_ci = ci([r['wq'] for r in runs])

    # P-K formula: Lq = (ρ²/(1-ρ)) * (1 + CV²) / 2
    if cv2 is not None:
        lq_pk = (RHO**2 / (1 - RHO)) * (1 + cv2) / 2
        wq_pk = lq_pk / LAMBDA
        pk_str = f'{lq_pk:>8.3f}'
        wq_pk_str = f'{wq_pk:>8.3f}'
    else:
        # Measure CV empirically for triangular
        rng_cv = random.Random(0)
        smp = [sampler(rng_cv) for _ in range(100_000)]
        cv  = statistics.stdev(smp) / statistics.mean(smp)
        lq_pk = (RHO**2 / (1 - RHO)) * (1 + cv**2) / 2
        wq_pk = lq_pk / LAMBDA
        pk_str    = f'{lq_pk:>8.3f}'
        wq_pk_str = f'{wq_pk:>8.3f}'

    sim_results[name] = dict(lq=lq_ci, wq=wq_ci, lq_pk=lq_pk)
    print(f'{name:<24} {lq_ci["mean"]:>8.3f} {lq_ci["half"]:>6.3f} {pk_str} {wq_ci["mean"]:>8.3f} {wq_pk_str}')

In [ ]:
names = list(sim_results.keys())
lq_means = [sim_results[n]['lq']['mean'] for n in names]
lq_ci_h  = [sim_results[n]['lq']['half'] for n in names]
lq_pk_v  = [sim_results[n]['lq_pk']      for n in names]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(x - 0.2, lq_means, 0.35, label='Simulation', color='steelblue', alpha=0.8,
              yerr=lq_ci_h, error_kw=dict(capsize=5))
ax.bar(x + 0.2, lq_pk_v, 0.35, label='P-K formula', color='coral', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=10)
ax.set_ylabel('Mean queue length Lq')
ax.set_title(f'Impact of service variability — ρ={RHO:.2f}, same mean service time')
ax.legend()

# Annotate with ratio vs constant
base_lq = lq_means[0]
for i, (mean, pk) in enumerate(zip(lq_means, lq_pk_v)):
    ax.text(i - 0.2, mean + 0.1, f'{mean/base_lq:.1f}×', ha='center', fontsize=9, color='steelblue')

plt.tight_layout()
plt.show()
print(f'Note: Constant service (M/D/1) has the minimum possible Lq = {lq_pk_v[0]:.3f}')
print(f'      Exponential service doubles Lq to ≈ {lq_pk_v[-1]:.3f} at the same load.')

## 4 · P-K Formula — Lq vs CV² at Fixed Load

Let's sweep CV from 0 to 2 and show how Lq scales linearly with `(1+CV²)/2`.

In [ ]:
cv2_range = np.linspace(0, 3, 100)
lq_pk_range = (RHO**2 / (1 - RHO)) * (1 + cv2_range) / 2

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cv2_range, lq_pk_range, 'b-', lw=2.5, label='P-K: Lq = (ρ²/(1-ρ)) · (1+CV²)/2')

# Mark specific distributions
markers = [
    ('Constant', 0.0,          'green'),
    ('Uniform',  1/3,          'orange'),
    ('Triangular', 0.37**2,    'purple'),
    ('Exponential', 1.0,       'red'),
]
for label, cv2, color in markers:
    lq_m = (RHO**2 / (1-RHO)) * (1 + cv2) / 2
    ax.scatter([cv2], [lq_m], color=color, s=80, zorder=5)
    ax.annotate(label, (cv2, lq_m), textcoords='offset points', xytext=(8, 2), fontsize=9, color=color)

ax.set_xlabel('CV² (squared coefficient of variation of service time)')
ax.set_ylabel('Mean queue length Lq')
ax.set_title(f'P-K formula — Lq linear in CV²  (ρ={RHO:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Summary

| Takeaway | Detail |
|----------|--------|
| **Variability = extra queue** | Same load, constant service → half the queue of exponential |
| **P-K formula** | Lq = (ρ²/(1-ρ)) · (1+CV²)/2 — exact for M/G/1 |
| **CV = σ/μ** | Measure it from your real data to predict queue length improvement |
| **Practical insight** | Standardising a process (reducing CV) is often cheaper than adding capacity |

**Next:** [Notebook 4](04_production_line.ipynb) — Serial multi-station production lines.